#P1: DESMUNTANT LA MENTIDA VISUAL

PREPOCESSAMENT DE DADES

In [72]:
import pandas as pd
import matplotlib.pyplot as plt
pd.set_option("display.max_colwidth", None)

In [73]:
df = pd.read_csv("dades/dades_grafic.csv")
df.head()

,Indicator Name,Indicator Code,Country Name,Country Code,Year,Value,Disaggregation
0,"Labor force participation rate, female (% of female population ages 15+) (modeled ILO estimate)",SL.TLF.CACT.FE.ZS,Africa Eastern and Southern,AFE,2025,63.674,"female, Modeled, 15+"
1,"Labor force participation rate, female (% of female population ages 15+) (modeled ILO estimate)",SL.TLF.CACT.FE.ZS,Africa Eastern and Southern,AFE,2024,63.643,"female, Modeled, 15+"
2,"Labor force participation rate, female (% of female population ages 15+) (modeled ILO estimate)",SL.TLF.CACT.FE.ZS,Africa Eastern and Southern,AFE,2023,63.593,"female, Modeled, 15+"
3,"Labor force participation rate, female (% of female population ages 15+) (modeled ILO estimate)",SL.TLF.CACT.FE.ZS,Africa Eastern and Southern,AFE,2022,59.831,"female, Modeled, 15+"
4,"Labor force participation rate, female (% of female population ages 15+) (modeled ILO estimate)",SL.TLF.CACT.FE.ZS,Africa Eastern and Southern,AFE,2021,59.454,"female, Modeled, 15+"


In [74]:
df.columns

Index(['Indicator Name', 'Indicator Code', 'Country Name', 'Country Code',
       'Year', 'Value', 'Disaggregation'],
      dtype='object')

El format del csv original era peculiar (ja que hi ha diferents tipus de Indicator Name), per tant, primer faig una adaptació. Se selecciona només les dades del 2023 i es guarda per cada país el % de dones i el % d'homes. 
Per tant, el primer que faig és començar a processar el dataframe però treballant només amb les dades del 2023, ja que és l'any en que es basa el gràfic.

In [75]:
df_2023 = df[df["Year"] == 2023]
df_2023.shape

(2841, 7)

El primer que volia comprovar era si el nombre de files era equiparable al nombre de països. Mirant per sobre el csv em vaig adonar que hi havia diferents entrades segons si es tractava del "Labor force participation rate" male i female. Per tant, el nombre de països hauria de ser la meitat del nombre de files. 


In [76]:
rows = len(df_2023)
total_indicator_names = len(df_2023["Indicator Name"].unique())
print(f"Hi ha {total_indicator_names} Indicator Names diferents")
total_countries = len(df_2023["Country Name"].unique())
print(f"Hi ha {total_countries} països i {rows} files")


Hi ha 15 Indicator Names diferents
Hi ha 236 països i 2841 files


Tal i com s'observa, hi ha una gran diferència respecte el nombre d'entrades i el nombre de països del dataset. 
A partir d'aquí, vaig decidir agrupar pels diferents "Indicator Name", per veure si en cadascun hi havia aquests 262 països

També, comprovo que hi hagi el mateix nombre de Country Names que de Country Codes. D'aquesta manera, comprovo que no hi hagi més d'un codi associat a cada país de daltaframe .

In [77]:
if(len(df_2023["Country Name"].unique())==len(df_2023["Country Code"].unique())):
    print("Cada país té un codi associat.")
else:
    print("Hi ha discrepàncies entre el nombre de països i el nombre de codis")

Cada país té un codi associat.


Comprovo que realment un mateix country code només està associat a un país

In [78]:

paisos_codis = df_2023.groupby("Country Name")["Country Code"].nunique()
if(max(paisos_codis)>1):
    print("Hi ha països associats a més d'un codi")
else:
    print("Cada país té un únic codi associat.")

Cada país té un únic codi associat.


In [79]:
indicador_names = df_2023.groupby("Indicator Name")
indicador_names.size()

Indicator Name
Labor force participation rate for ages 15-24, female (%) (modeled ILO estimate)                     231
Labor force participation rate for ages 15-24, female (%) (national estimate)                        127
Labor force participation rate for ages 15-24, male (%) (modeled ILO estimate)                       231
Labor force participation rate for ages 15-24, male (%) (national estimate)                          127
Labor force participation rate for ages 15-24, total (%) (modeled ILO estimate)                      231
Labor force participation rate for ages 15-24, total (%) (national estimate)                         127
Labor force participation rate, female (% of female population ages 15+) (modeled ILO estimate)      231
Labor force participation rate, female (% of female population ages 15+) (national estimate)         127
Labor force participation rate, female (% of female population ages 15-64) (modeled ILO estimate)    231
Labor force participation rate, male (% 

En el gràfic posa escrit "Figures are rounded and based on the International Labour Organization's estimates as of 2023", per tant, treballaré amb aquells que indiquin "modeled ILO estimate". A partir d'aquí, cal averiguar quin és el rang d'edat (15-24, 15-64 o 15+) que es mostra al gràfic (ja que no està indicat).
Per comprovar quin rang d'edat s'utilitza en el gràfic, comparo les dades d'Afganistan per als diferents intervals disponibles. El gràfic mostra un 100% de participació laboral masculina i un 5% femenina, per tant, busco quin dels indicadors s'aproxima a aquests valors.

In [81]:
afghanistan=df_2023[df_2023["Country Name"] == "Afghanistan"]
afghanistan=afghanistan[afghanistan["Indicator Name"].str.contains("ILO estimate")]
afghanistan = afghanistan[["Indicator Name","Value"]]
afghanistan

,Indicator Name,Value
1694,"Labor force participation rate, female (% of female population ages 15+) (modeled ILO estimate)",5.145
15611,"Labor force participation rate, male (% of male population ages 15+) (modeled ILO estimate)",70.074
29527,"Labor force participation rate, total (% of total population ages 15+) (modeled ILO estimate)",37.588
43670,"Labor force participation rate for ages 15-24, female (%) (modeled ILO estimate)",6.013
56953,"Labor force participation rate for ages 15-24, male (%) (modeled ILO estimate)",68.598
70236,"Labor force participation rate for ages 15-24, total (%) (modeled ILO estimate)",38.006
83544,"Labor force participation rate, female (% of female population ages 15-64) (modeled ILO estimate)",5.359
91954,"Labor force participation rate, male (% of male population ages 15-64) (modeled ILO estimate)",71.384
100364,"Labor force participation rate, total (% of total population ages 15-64) (modeled ILO estimate)",38.567


Tal i com s'observa, en el % d'homes en cap de les tres opcions coincideix amb el 100% que es mostra el gràfic, ja que els valors se situen entre el 68,598% i el 71,384%.
En canvi, en el % de dones, sí que hi ha el 5% en les entrades on l'edat és de 15-64 o de 15+. 
Per tant, miro pels 12 paísos del gràfic tots els valors, per veure si és un cas aïllat o és per tots els països.

In [85]:
paisos = [
    "Afghanistan",
    "Iraq",
    "Pakistan",
    "Oman",
    "Iran",
    "Yemen",
    "Egypt",
    "Algeria",
    "Syria",
    "Jordan",
    "Morocco",
    "Saudi Arabia"
]
df_12 = df_2023[df_2023["Country Name"].isin(paisos)]
df_12 = df_12[df_12["Indicator Name"].str.contains("modeled ILO estimate")]
df_12_15 = df_12[df_12["Indicator Name"].str.contains("population ages 15+",regex=False)]
df_12_15 = df_12_15.sort_values(["Country Name", "Indicator Name"])
df_12_15[["Country Name", "Indicator Name", "Value"]]


,Country Name,Indicator Name,Value
1694,Afghanistan,"Labor force participation rate, female (% of female population ages 15+) (modeled ILO estimate)",5.145
15611,Afghanistan,"Labor force participation rate, male (% of male population ages 15+) (modeled ILO estimate)",70.074
29527,Afghanistan,"Labor force participation rate, total (% of total population ages 15+) (modeled ILO estimate)",37.588
1766,Algeria,"Labor force participation rate, female (% of female population ages 15+) (modeled ILO estimate)",14.338
15683,Algeria,"Labor force participation rate, male (% of male population ages 15+) (modeled ILO estimate)",67.610
29599,Algeria,"Labor force participation rate, total (% of total population ages 15+) (modeled ILO estimate)",41.507
4538,Iraq,"Labor force participation rate, female (% of female population ages 15+) (modeled ILO estimate)",10.913
18455,Iraq,"Labor force participation rate, male (% of male population ages 15+) (modeled ILO estimate)",72.924
32371,Iraq,"Labor force participation rate, total (% of total population ages 15+) (modeled ILO estimate)",41.628
4754,Jordan,"Labor force participation rate, female (% of female population ages 15+) (modeled ILO estimate)",15.933


In [ ]:
afghanistan=df_2023[df_2023["Country Name"] == "Afghanistan"]
afghanistan=afghanistan[afghanistan["Indicator Name"].str.contains("ILO estimate")]
afghanistan = afghanistan[["Indicator Name","Value"]]
afghanistan

,Indicator Name,Value
1694,"Labor force participation rate, female (% of female population ages 15+) (modeled ILO estimate)",5.145
15611,"Labor force participation rate, male (% of male population ages 15+) (modeled ILO estimate)",70.074
29527,"Labor force participation rate, total (% of total population ages 15+) (modeled ILO estimate)",37.588
43670,"Labor force participation rate for ages 15-24, female (%) (modeled ILO estimate)",6.013
56953,"Labor force participation rate for ages 15-24, male (%) (modeled ILO estimate)",68.598
70236,"Labor force participation rate for ages 15-24, total (%) (modeled ILO estimate)",38.006
83544,"Labor force participation rate, female (% of female population ages 15-64) (modeled ILO estimate)",5.359
91954,"Labor force participation rate, male (% of male population ages 15-64) (modeled ILO estimate)",71.384
100364,"Labor force participation rate, total (% of total population ages 15-64) (modeled ILO estimate)",38.567


Tal i com s'observa, alguns valors coincideixen, però no tots. Per tant, dedueixo que s'ha fet alguna adaptació amb els percentatges.
Són amb aquests valors que contrueixo un nou dataframe transposat per poder després fer el gràfic. 

In [ ]:
df_grafic = df_12_15.pivot(
    index="Country Name",
    columns="Indicator Name",
    values="Value"
)
df_grafic = df_grafic.rename(columns={
    "Labor force participation rate, female (% of female population ages 15+) (modeled ILO estimate)": "Female",
    "Labor force participation rate, male (% of male population ages 15+) (modeled ILO estimate)": "Male",
    "Labor force participation rate, total (% of total population ages 15+) (modeled ILO estimate)": "Total"
})

df_grafic

Indicator Name,Female,Male,Total
Country Name,,,
Afghanistan,5.145,70.074,37.588
Algeria,14.338,67.610,41.507
Iraq,10.913,72.924,41.628
Jordan,15.933,62.524,40.123
Morocco,19.671,68.622,44.224
Oman,30.321,88.060,68.331
Pakistan,23.997,80.077,52.372
Saudi Arabia,34.278,83.024,65.336
